In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, TimestampType
)
from datetime import datetime

# Testdaten mit absichtlichen Fehlern
test_data = [
    (1001, "ECU_Engine", "P0300",
     "Short_Circuit", "PASS",
     datetime(2026, 10, 20, 8, 30)),

    (1002, "ECU_Battery", "P0A80",
     "Voltage_Deviation", "FAIL",
     datetime(2026, 10, 20, 9, 15)),

    (1002, "ECU_Battery", "P0A80",
     "Voltage_Deviation", "PASS",
     datetime(2026, 10, 20, 9, 20)),

    (None, "ECU_Brake", "C0035",
     "Sensor_Failure", "ERROR",
     datetime(2026, 10, 20, 10, 0)),

    (1005, "ECU_Engine", "INVALID",
     "Unknown_Category", "FAIL",
     datetime(2026, 10, 20, 11, 0))
]

schema = StructType([
    StructField("TestID", IntegerType(), True),
    StructField("ECU", StringType(), True),
    StructField("DTC", StringType(), True),
    StructField("ErrorCategory", StringType(), True),
    StructField("TestResult", StringType(), True),
    StructField("TestTimestamp", TimestampType(), True)
])

df_negative = spark.createDataFrame(
    test_data, schema
)

print("Negativtest-Datensätze:", df_negative.count())

display(df_negative)


StatementMeta(, b6ea1b56-e928-4823-ba2a-629be50c7de1, 3, Finished, Available, Finished, False)

Negativtest-Datensätze: 5


SynapseWidget(Synapse.DataFrame, f8c37ce0-45a6-45ba-8f7b-fea627029676)

In [2]:
from pyspark.sql import functions as F

# Negativtest-Daten verwenden
df = df_negative

# 1. Anzahl Datensätze
total_rows = df.count()

# 2. Fehlende Pflichtfelder
required_columns = [
    "TestID", "ECU", "DTC",
    "ErrorCategory", "TestResult",
    "TestTimestamp"
]

missing_condition = F.lit(False)

for col_name in required_columns:
    missing_condition = (
        missing_condition | F.col(col_name).isNull()
    )

missing_rows = df.filter(missing_condition).count()

# 3. Doppelte TestIDs
duplicate_ids = (
    df.filter(F.col("TestID").isNotNull())
    .groupBy("TestID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 4. Ungültige Testergebnisse
invalid_results = df.filter(
    F.col("TestResult").isNull()
    | ~F.upper(F.trim(F.col("TestResult")))
        .isin("PASS", "FAIL")
).count()

# 5. Ungültige DTC-Codes
invalid_dtcs = df.filter(
    F.col("DTC").isNull()
    | ~F.upper(F.trim(F.col("DTC")))
        .rlike("^[PCBU][0-9A-F]{4}$")
).count()

# 6. Ungültige Fehlerkategorien
valid_categories = [
    "Short_Circuit",
    "Overtemperature",
    "Communication_Loss",
    "Voltage_Deviation",
    "Sensor_Failure",
    "Diagnostic_Error"
]

invalid_categories = df.filter(
    F.col("ErrorCategory").isNull()
    | ~F.trim(F.col("ErrorCategory"))
        .isin(valid_categories)
).count()

# 7. Gesamtergebnis
quality_status = (
    "PASS"
    if total_rows == 1000
    and missing_rows == 0
    and duplicate_ids == 0
    and invalid_results == 0
    and invalid_dtcs == 0
    and invalid_categories == 0
    else "FAIL"
)

print("=== NEGATIVE DATA QUALITY TEST ===")
print("TotalRows:", total_rows)
print("MissingRows:", missing_rows)
print("DuplicateTestIDs:", duplicate_ids)
print("InvalidResults:", invalid_results)
print("InvalidDTCs:", invalid_dtcs)
print("InvalidCategories:", invalid_categories)
print("QualityStatus:", quality_status)


StatementMeta(, b6ea1b56-e928-4823-ba2a-629be50c7de1, 4, Finished, Available, Finished, False)

=== NEGATIVE DATA QUALITY TEST ===
TotalRows: 5
MissingRows: 1
DuplicateTestIDs: 1
InvalidResults: 1
InvalidDTCs: 1
InvalidCategories: 1
QualityStatus: FAIL


In [3]:
# Schritt 43.8.4 – Negativtest des Quality Gates

print("=== NEGATIVE QUALITY GATE TEST ===")

if quality_status == "FAIL":
    print("Quality Gate: FAILED")
    print("Fehlerhafte Daten erkannt!")
    print("Die weitere Verarbeitung wird blockiert.")

    raise ValueError(
        "Data Quality Gate FAILED: "
        "Ungültige Automotive-Testdaten erkannt."
    )

else:
    print("Quality Gate: PASSED")
    print("Die Datenverarbeitung ist freigegeben.")


StatementMeta(, b6ea1b56-e928-4823-ba2a-629be50c7de1, 5, Finished, Available, Finished, False)

=== NEGATIVE QUALITY GATE TEST ===
Quality Gate: FAILED
Fehlerhafte Daten erkannt!
Die weitere Verarbeitung wird blockiert.


ValueError: Data Quality Gate FAILED: Ungültige Automotive-Testdaten erkannt.

## Data Quality – Negative Test

### Test Objective
Verify that invalid automotive diagnostic data
is detected and rejected by the Data Quality Gate.

### Test Dataset
- Total records: 5
- Missing TestID: 1
- Duplicate TestID group: 1
- Invalid TestResult: 1
- Invalid DTC: 1
- Invalid ErrorCategory: 1

### Expected Result
QualityStatus: FAIL

The notebook must raise a ValueError when
the Data Quality Gate detects invalid data.

### Actual Result
QualityStatus: FAIL
Quality Gate: FAILED
ValueError raised as expected.

### Test Status
PASSED – The negative test behaved as expected.

### Scope
This test validates the notebook's failure
mechanism using an isolated DataFrame.

The production Bronze, Silver and Gold tables
were not modified.

A pipeline-level negative integration test
is still required.

## Pipeline Integration Test – Negative Scenario

### Objective
Verify that an invalid dataset causes the Data
Quality Gate to fail and prevents downstream
pipeline activities from running.

### Test Pipeline
PL_Data_Quality_Negative_Test

### Test Configuration
1. ACT_01_Negative_Quality_Gate
   - Notebook: NB_10_Data_Quality_Negative_Test
   - Expected: FAILED

2. ACT_02_Should_Not_Run
   - Activity: Wait (5 seconds)
   - Dependency: On Success
   - Expected: Not executed

### Actual Results
- Invalid records detected: YES
- QualityStatus: FAIL
- ValueError raised: YES
- Quality Gate activity: FAILED
- Pipeline status: FAILED
- Downstream activity: Not shown in run history
- On-Success dependency: Confirmed

### Test Conclusion
PASSED – The Quality Gate failed as expected.
The downstream Wait activity was not observed
executing. Its On-Success dependency was
confirmed.

### Safety
Production Bronze, Silver and Gold tables
were not modified by this negative test.
